# Section 1: Research - Historical & Advanced Classical Ciphers


## The Vigenère Cipher

The Vigenère cipher is a polyalphabetic substitution cipher. Which means that it uses multiple substitution alphabets instead of just one, switching between them throughtout the message. 

### Mechanism: 

The Vigenère cipher uses a Tabula Recta - a 26×26 grid where:
- Each row represents a Caesar cipher with a different shift
- Row 0: A=A, B=B, C=C... (shift 0)
- Row 1: A=B, B=C, C=D... (shift 1)
- Row 2: A=C, B=D, C=E... (shift 2)
- And so on...

**Encryption Process:**
1. Choose a keyword (e.g., "KEY")
2. Repeat the keyword to match the plaintext length
3. For each letter pair (plaintext, keyword), find the intersection in the Tabula Recta

**Example:**
- Plaintext: "HELLO"
- Keyword: "KEY" → "KEYKE"
- H+K=R, E+E=I, L+Y=J, L+K=V, O+E=S
- Ciphertext: "RIJVS"

In [2]:
vocab = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Encryption
def vigenere_encrypt(plaintext, keyword):
    result = ""
    keyword = keyword.upper()
    plaintext = plaintext.upper()
    
    for i, char in enumerate(plaintext):
        if char in vocab:
            char_pos = vocab.index(char)
            key_pos = vocab.index(keyword[i % len(keyword)])
            encrypted_pos = (char_pos + key_pos) % len(vocab)
            result += vocab[encrypted_pos]
        else:
            result += char
    
    return result

# Decryption
def vigenere_decrypt(ciphertext, keyword):
    result = ""
    keyword = keyword.upper()
    ciphertext = ciphertext.upper()
    
    for i, char in enumerate(ciphertext):
        if char in vocab:
            char_pos = vocab.index(char)
            key_pos = vocab.index(keyword[i % len(keyword)])
            decrypted_pos = (char_pos - key_pos) % len(vocab)
            result += vocab[decrypted_pos]
        else:
            result += char
    
    return result

# Example
plaintext = "HELLO"
keyword = "KEY"
ciphertext = vigenere_encrypt(plaintext, keyword)
decrypted = vigenere_decrypt(ciphertext, keyword)

print(f"Plaintext:  {plaintext}")
print(f"Keyword:    {keyword}")
print(f"Ciphertext: {ciphertext}")
print(f"Decrypted:  {decrypted}")

Plaintext:  HELLO
Keyword:    KEY
Ciphertext: RIJVS
Decrypted:  HELLO


### How does this polyalphabetic approach defend against basic frequency analysis?

Each letter in the plaintext can be encrypted to different ciphertext letters depending on its position, taking that into account, common letters like 'E' don't always map to the same ciphertext letter. Also, the longer the key, the more scrambled the frequency distribution becomes

**Kasiski Examination** is a method to find the keyword length:

1. **Find Repeated Patterns**: Look for identical sequences in the ciphertext
2. **Measure Distances**: Calculate the distance between these repetitions
3. **Find GCD**: The greatest common divisor of these distances likely reveals the keyword length
4. **Frequency Analysis**: Once the keyword length is known, perform frequency analysis on each "column" (every nth letter)


## The Hill Cipher

The Hill cipher is a polygraphic substitution cipher based on linear algebra, invented by Lester S. Hill in 1929.

### Mathematical Basis

The Hill cipher converts plaintext into numerical vectors and uses matrix multiplication for encryption:

1. Convert plaintext blocks into numerical vectors (A=0, B=1, ..., Z=25)
2. Multiply the plaintext vector by a key matrix (row linear combination)
3. Apply modulo 26 to keep results in the alphabet range
4. Convert the result back to letters

C = K × P (mod 26)

Where:
- C = ciphertext vector
- K = key matrix (n×n)
- P = plaintext vector (n×1)

In [3]:
# Encryption
def hill_encrypt(plaintext, key_matrix):
    plaintext = plaintext.upper().replace(' ', '')
    n = len(key_matrix)
    
    while len(plaintext) % n != 0:
        plaintext += 'X'
    
    ciphertext = ""
    for i in range(0, len(plaintext), n):
        block = plaintext[i:i+n]
        
        numbers = []
        for char in block:
            numbers.append(vocab.index(char))
        
        # Algebra
        encrypted_numbers = []
        for row in key_matrix:
            total = 0
            for j in range(len(numbers)):
                total += row[j] * numbers[j]
            encrypted_numbers.append(total % 26)
        
        for num in encrypted_numbers:
            ciphertext += vocab[num]
    
    return ciphertext


### Mathematical Requirements for the Key Matrix

1. The matrix must have a multiplicative inverse modulo 26
2. The determinant of the matrix must be coprime to 26 (gcd(det(K), 26) = 1)
3.  Since 26 = 2 × 13, the determinant cannot be divisible by 2 or 13

**Valid determinants modulo 26**: 1, 3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25
```
P = K⁻¹ × C (mod 26)
```

In [4]:
# Decryption for a 2x2 matrix
def hill_decrypt(ciphertext, key_matrix):
    if len(key_matrix) == 2:
        det = (key_matrix[0][0] * key_matrix[1][1] - key_matrix[0][1] * key_matrix[1][0]) % 26
        det_inv = None
        for i in range(26):
            if (det * i) % 26 == 1:
                det_inv = i
                break
        
        if det_inv is None:
            return "Error: No invertible matrix"
        
        inv_matrix = [
            [(key_matrix[1][1] * det_inv) % 26, (-key_matrix[0][1] * det_inv) % 26],
            [(-key_matrix[1][0] * det_inv) % 26, (key_matrix[0][0] * det_inv) % 26]
        ]
        
        return hill_encrypt(ciphertext, inv_matrix)
    else:
        return "Invalid key matrix size"

# Example with 2x2 matrix
key_matrix = [[3, 2], [5, 7]]
plaintext = "HELP"
ciphertext = hill_encrypt(plaintext, key_matrix)
decrypted = hill_decrypt(ciphertext, key_matrix)

print(f"Key Matrix: {key_matrix}")
print(f"Plaintext:  {plaintext}")
print(f"Ciphertext: {ciphertext}")
print(f"Decrypted:  {decrypted}")

Key Matrix: [[3, 2], [5, 7]]
Plaintext:  HELP
Ciphertext: DLLE
Decrypted:  HELP


## 3. The Playfair Cipher

The Playfair cipher is a digraph substitution cipher, encrypting pairs of letters instead of individual letters.

In [5]:

def create_playfair_grid(keyword):
    keyword = keyword.upper().replace('J', 'I')  
    alphabet = "ABCDEFGHIKLMNOPQRSTUVWXYZ"  
    

    key = ""
    for char in keyword:
        if char not in key and char in alphabet:
            key += char
    

    for char in alphabet:
        if char not in key:
            key += char

    grid = []
    for i in range(0, 25, 5):
        row = []
        for j in range(5):
            row.append(key[i + j])
        grid.append(row)
    
    return grid

def find_position(grid, letter):
    for i in range(5):
        for j in range(5):
            if grid[i][j] == letter:
                return (i, j)
    return None

def clean_plaintext(text):

    result = ""
    i = 0
    while i < len(text):
        if i < len(text) - 1:
            if text[i] == 'X' and i > 0 and i < len(text) - 1:
                if text[i-1] == text[i+1]:
                    i += 1
                    continue
        result += text[i]
        i += 1
    

    if result.endswith('X') and len(result) > 1:
        result = result[:-1]
    
    return result

def playfair_encrypt(plaintext, keyword):
    grid = create_playfair_grid(keyword)
    plaintext = plaintext.upper().replace('J', 'I').replace(' ', '')
    

    pairs = []
    i = 0
    while i < len(plaintext):
        if i + 1 < len(plaintext):
            if plaintext[i] == plaintext[i + 1]:  
                pairs.append(plaintext[i] + 'X')
                i += 1
            else:
                pairs.append(plaintext[i:i+2])
                i += 2
        else:
            pairs.append(plaintext[i] + 'X')  
            i += 1
    
    ciphertext = ""
    for pair in pairs:
        pos1 = find_position(grid, pair[0])
        pos2 = find_position(grid, pair[1])
        
        if pos1[0] == pos2[0]:  
            new_pos1 = (pos1[0], (pos1[1] + 1) % 5)
            new_pos2 = (pos2[0], (pos2[1] + 1) % 5)
        elif pos1[1] == pos2[1]:  
            new_pos1 = ((pos1[0] + 1) % 5, pos1[1])
            new_pos2 = ((pos2[0] + 1) % 5, pos2[1])
        else:  
            new_pos1 = (pos1[0], pos2[1])
            new_pos2 = (pos2[0], pos1[1])
        
        ciphertext += grid[new_pos1[0]][new_pos1[1]] + grid[new_pos2[0]][new_pos2[1]]
    
    return ciphertext

def playfair_decrypt(ciphertext, keyword):
    grid = create_playfair_grid(keyword)
    

    pairs = []
    for i in range(0, len(ciphertext), 2):
        pairs.append(ciphertext[i:i+2])
    
    plaintext = ""
    for pair in pairs:
        pos1 = find_position(grid, pair[0])
        pos2 = find_position(grid, pair[1])
        
        if pos1[0] == pos2[0]:  
            new_pos1 = (pos1[0], (pos1[1] - 1) % 5)
            new_pos2 = (pos2[0], (pos2[1] - 1) % 5)
        elif pos1[1] == pos2[1]:  # 
            new_pos1 = ((pos1[0] - 1) % 5, pos1[1])
            new_pos2 = ((pos2[0] - 1) % 5, pos2[1])
        else:  
            new_pos1 = (pos1[0], pos2[1])
            new_pos2 = (pos2[0], pos1[1])
        
        plaintext += grid[new_pos1[0]][new_pos1[1]] + grid[new_pos2[0]][new_pos2[1]]
    

    return clean_plaintext(plaintext)

# Example
keyword = "MONARCHY"
grid = create_playfair_grid(keyword)
print("Playfair Grid:")
for row in grid:
    print(' '.join(row))

plaintext = "HELLO"
ciphertext = playfair_encrypt(plaintext, keyword)
decrypted = playfair_decrypt(ciphertext, keyword)

print(f"\\nPlaintext:  {plaintext}")
print(f"Ciphertext: {ciphertext}")
print(f"Decrypted:  {decrypted}")


Playfair Grid:
M O N A R
C H Y B D
E F G I K
L P Q S T
U V W X Z
\nPlaintext:  HELLO
Ciphertext: CFSUPM
Decrypted:  HELLO


### Historical Context: Why was it considered "field-ready" compared to more complex systems?

**World War I Usage:**
- British forces extensively used Playfair for tactical communications
- Replaced simpler substitution ciphers that were easily broken
- Used in trenches for coordinating attacks and supply movements

**World War II Usage:**
- Continued use by British and Commonwealth forces
- Used alongside more complex systems for different security levels
- Employed in resistance movements due to its simplicity

No mechanical devices required. It was faster than complex mathematical ciphers (required only a small card or could be drawn quickly) and the grid could be memorized and reconstructed from keyword. It was one of the first tool with error tolerance because a single letter transmission error did not cascade.

## 4. The Enigma Machine

The Enigma machine was an electro-mechanical rotor cipher machine used by Nazi Germany during World War II.

### Rotor Logic: Dynamic Substitution

**Rotor Mechanism:**
- Each rotor contains 26 electrical contacts (one for each letter)
- Internal wiring creates a substitution cipher
- **Key Innovation**: Rotors advance with each keystroke, changing the substitution alphabet

**Rotor Movement:**
1. **Rightmost rotor** advances with every keystroke
2. **Middle rotor** advances when the rightmost reaches its turnover position
3. **Leftmost rotor** advances when the middle reaches its turnover position
4. **Double-stepping**: Middle rotor advances twice in sequence during turnover

**Why This Was Revolutionary:**
- The substitution alphabet changed for EVERY single letter
- Even typing "AAAAAAA" would produce completely different ciphertext
- Made traditional frequency analysis nearly impossible

### The Reflector: Ensuring Reciprocal Encryption

**Purpose of the Reflector:**
1. **Signal Return**: After passing through all rotors, the signal hits the reflector and returns through the rotors in reverse
2. **Reciprocal Operation**: Made encryption and decryption the same operation
3. **Operational Simplicity**: Operators didn't need separate encrypt/decrypt procedures

**Critical Weakness - Self-Reciprocal Property:**
- **No letter could encrypt to itself** due to the reflector design
- If you typed 'E', you could never get 'E' as output
- This gave codebreakers a crucial constraint: eliminated 1/26 of possibilities for each position
- Helped in crib analysis (matching known plaintext patterns)

**Mathematical Impact:**
- Reduced effective keyspace by eliminating impossible configurations
- Enabled "cribbing" attacks where known plaintext could be matched against ciphertext

### The Plugboard (Steckerbrett): Exponential Key Expansion

**Plugboard Mechanism:**
- 26 sockets on the front of the machine (one per letter)
- Cables could connect pairs of letters, swapping them before and after rotor encryption
- Standard operation used 10 cables (20 letters), leaving 6 letters unchanged

**Exponential Increase in Configurations:**
1. **Without Plugboard**: ~10¹⁶ possible rotor configurations
2. **With Plugboard**: Added approximately 10¹⁴ additional configurations
3. **Total Keyspace**: ~10²³ possible configurations

**Mathematical Calculation of Plugboard Configurations:**
- Number of ways to choose 20 letters from 26: C(26,20)
- Number of ways to pair 20 letters: (20!)/(2¹⁰ × 10!)
- Total plugboard settings: ~150 trillion

**Why This Was Formidable:**
- Even if codebreakers solved the rotor settings, they still faced the plugboard
- Made brute force attacks computationally infeasible with 1940s technology
- Required sophisticated mathematical techniques (developed at Bletchley Park) to overcome